# Evaluation — Brain Tissue Dice Scores

Evaluates model quality by:
1. Running inference on the 106-scan test set (same split as training notebooks)
2. Segmenting predictions and real HUNT4 images with FastSurfer
3. Computing Dice scores for Gray Matter (GM), White Matter (WM), and CSF

**Segmentations are cached** — re-running a model or switching models skips already-computed files.

## 1. Bootstrap

In [135]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [136]:
import os
import subprocess
import sys

FASTSURFER_DIR = os.path.join(os.path.expanduser("~"), "FastSurfer")

if not os.path.isdir(FASTSURFER_DIR):
    print("Cloning FastSurfer...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/Deep-MI/FastSurfer.git", FASTSURFER_DIR],
        check=True
    )
    print("FastSurfer cloned.")
else:
    print("FastSurfer already available.")

# _meta_registrations.py tries to register fake kernels for torchvision ops
# (e.g. torchvision::nms) at import time. When torchvision C extension is
# incompatible with the installed torch, this raises RuntimeError and blocks
# all torchvision imports. FastSurfer only needs torchvision.transforms, not
# NMS or any detection ops, so blanking _meta_registrations.py is safe.
import importlib.util
spec = importlib.util.find_spec("torchvision")
if spec is not None:
    meta_path = os.path.join(os.path.dirname(spec.origin), "_meta_registrations.py")
    if os.path.isfile(meta_path):
        with open(meta_path) as _f:
            _existing = _f.read()
        if "# PATCHED" not in _existing:
            with open(meta_path, "w") as _f:
                _f.write("# PATCHED - disabled for Python 3.14 / torch compatibility\n")
            print("Patched torchvision._meta_registrations.")
        else:
            print("torchvision._meta_registrations already patched.")

# Verify fix in a subprocess
verify = subprocess.run(
    [sys.executable, "-c", "import torchvision; print(torchvision.__version__)"],
    capture_output=True, text=True
)
if verify.returncode != 0:
    print("[WARN] torchvision still broken: " + verify.stderr[:500])
else:
    print("torchvision " + verify.stdout.strip() + " OK.")

print("Bootstrap done.")


FastSurfer already available.
torchvision._meta_registrations already patched.
torchvision 0.26.0+cu130 OK.
Bootstrap done.


In [137]:
import numpy as np
import nibabel as nib
import torch
import pandas as pd
from tqdm import tqdm

from utils.data_loader import DataLoader
from utils.metadata_loader import MetaDataLoader
from utils.data_converter import DataConverter
from models.unet_3d_film import FiLMUNet3D
from models.unet_3d_std import UNet3D

## 2. Configuration

Edit **only this cell** to switch between models.

In [138]:
# ============================================================
# USER CONFIGURATION — edit these lines to switch models
# ============================================================
MODEL_NAME = "standard_testloss_0.0528"   # Used for output directory names
MODEL_PATH = "out/unets/3d_unet_model_standard_testloss_0.0528.pt"
MODEL_TYPE = "standard"   # "film_complex" | "film_simple" | "standard"
#
# Other available models:
#   MODEL_NAME = "film_simple_testloss_0.0517"
#   MODEL_PATH = "out/unets/3d_unet_model_film_simple_testloss_0.0517.pt"
#   MODEL_TYPE = "film_simple"
#
#   MODEL_NAME = "standard"
#   MODEL_PATH = "out/unets/3d_unet_model_standard_testloss_0.0528.pt"
#   MODEL_TYPE = "standard"
# ============================================================

# Fixed constants
CROP_AXES = ((16, 10, 0), (17, 11, 17))
SEED      = 69
THRESHOLD = 1e-3   # Prediction values with abs < THRESHOLD are zeroed out

# Derived output paths
PRED_DIR     = f"out/{MODEL_NAME}/predictions"
PRED_SEG_DIR = f"out/{MODEL_NAME}/segmentations"
GT_SEG_DIR   = "out/hunt4_gt/segmentations"  # shared across all models

os.makedirs(PRED_DIR,     exist_ok=True)
os.makedirs(PRED_SEG_DIR, exist_ok=True)
os.makedirs(GT_SEG_DIR,   exist_ok=True)

print(f"Model      : {MODEL_NAME}  ({MODEL_TYPE})")
print(f"Predictions: {PRED_DIR}")
print(f"Pred segs  : {PRED_SEG_DIR}")
print(f"GT segs    : {GT_SEG_DIR}")

Model      : standard_testloss_0.0528  (standard)
Predictions: out/standard_testloss_0.0528/predictions
Pred segs  : out/standard_testloss_0.0528/segmentations
GT segs    : out/hunt4_gt/segmentations


## 3. Test Split & Model Loading

In [139]:
# Reproduce the exact same 106-patient test split used during training
data_loader = DataLoader(root_path="data/")
_, _, test_pairs = data_loader.split_dataset_paths(seed=SEED)
print(f"Test set: {len(test_pairs)} patients")

Test set: 106 patients


In [140]:
if MODEL_TYPE == "film_complex":
    model = FiLMUNet3D(in_ch=1, base=32, cond_dim=3, use_simple=False)
elif MODEL_TYPE == "film_simple":
    model = FiLMUNet3D(in_ch=1, base=32, cond_dim=3, use_simple=True)
elif MODEL_TYPE == "standard":
    model = UNet3D(in_ch=1, out_ch=1, base=32)
else:
    raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE!r}")

model.load_state_dict(torch.load(MODEL_PATH, weights_only=False))
model.eval()
print(f"Loaded: {MODEL_PATH}")

Loaded: out/unets/3d_unet_model_standard_testloss_0.0528.pt


## 4. Inference — Predict HUNT4-like T1 Images

Saves each prediction as a full-size (193×229×193) NIfTI in `out/{model_name}/predictions/`.  
Skips patients whose prediction file already exists.

In [141]:
dc = DataConverter()
ml = MetaDataLoader()


def run_inference_and_save(hunt3_path, hunt4_path, out_path):
    """Run model on one HUNT3 scan and save the predicted HUNT4-like volume."""
    # Load and crop input
    x = dc.load_path_as_tensor(hunt3_path, device="cuda:0")
    x_crop = dc.get_volume_with_3d_change(x, CROP_AXES, remove_mode=True)

    with torch.no_grad():
        if MODEL_TYPE in ("film_complex", "film_simple"):
            cond = ml.get(hunt3_path, sex=True, age_hunt3=True, age_hunt4=True)
            cond_t = torch.tensor(cond, dtype=torch.float32).unsqueeze(0).to("cuda:0")
            pred = model(x_crop, cond_t)
        else:
            pred = model(x_crop)

    pred[pred.abs() < THRESHOLD] = 0.0

    # Both model types return output on cuda:1 — move to CPU before padding
    # Uncrop: zero-pad back to full MNI volume (193×229×193)
    pred_full = dc.get_volume_with_3d_change(pred.to("cpu"), CROP_AXES, remove_mode=False)
    pred_np = pred_full.squeeze().numpy()

    # Save with the real MNI affine from the HUNT4 reference image
    ref = nib.load(hunt4_path)
    nib.save(
        nib.Nifti1Image(pred_np.astype(np.float32), ref.affine, ref.header),
        out_path
    )

In [142]:
n_skipped = 0
for hunt3_path, hunt4_path in tqdm(test_pairs, desc="Running inference"):
    patient_id = os.path.basename(os.path.dirname(hunt3_path))
    pred_path  = os.path.join(PRED_DIR, f"{patient_id}_pred.nii.gz")

    if os.path.isfile(pred_path):
        n_skipped += 1
        continue

    run_inference_and_save(hunt3_path, hunt4_path, pred_path)
    torch.cuda.empty_cache()

print(f"Inference done. ({n_skipped} skipped — already existed)")

Running inference: 100%|██████████| 106/106 [02:32<00:00,  1.44s/it]

Inference done. (0 skipped — already existed)


## 5. FastSurfer Segmentation

Runs FastSurfer's CNN segmentation (DKT parcellation) on:
- **Model predictions** → `out/{model_name}/segmentations/{patient_id}/pred_seg.nii.gz`
- **HUNT4 ground truth** → `out/hunt4_gt/segmentations/{patient_id}/gt_seg.nii.gz` *(shared across models)*

Each call is skipped if the output file already exists.

In [143]:
def run_fastsurfer_seg(input_path, output_path):
    """Run FastSurfer CNN segmentation. Returns True on success."""
    # Derive subject dir and id from output path structure:
    # output_path = .../segmentations/{sid}/pred_seg.nii.gz
    abs_out  = os.path.abspath(output_path)
    sid      = os.path.basename(os.path.dirname(abs_out))
    sd       = os.path.dirname(os.path.dirname(abs_out))
    os.makedirs(os.path.dirname(abs_out), exist_ok=True)
    cmd = [
        sys.executable,
        os.path.join(FASTSURFER_DIR, "FastSurferCNN", "run_prediction.py"),
        "--t1",              os.path.abspath(input_path),
        "--asegdkt_segfile", abs_out,
        "--sd",              sd,
        "--sid",             sid,
        "--device",          "cuda",
        "--batch_size",      "1",
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = FASTSURFER_DIR + os.pathsep + env.get("PYTHONPATH", "")
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print("\n[ERROR] FastSurfer failed on " + input_path)
        print(result.stderr[-1500:])
        return False
    return True


In [144]:
# Segment model predictions
n_skipped = 0
for hunt3_path, _ in tqdm(test_pairs, desc="Segmenting predictions"):
    patient_id = os.path.basename(os.path.dirname(hunt3_path))
    pred_path  = os.path.join(PRED_DIR, f"{patient_id}_pred.nii.gz")
    seg_path   = os.path.join(PRED_SEG_DIR, patient_id, "pred_seg.nii.gz")

    if os.path.isfile(seg_path):
        n_skipped += 1
        continue

    if not os.path.isfile(pred_path):
        print(f"[WARN] No prediction found for {patient_id} — skipping segmentation")
        continue

    run_fastsurfer_seg(pred_path, seg_path)

print(f"Prediction segmentation done. ({n_skipped} skipped)")

Segmenting predictions: 100%|██████████| 106/106 [56:49<00:00, 32.16s/it]

Prediction segmentation done. (0 skipped)


In [145]:
# Segment HUNT4 ground truth (shared — only computed once across all models)
n_skipped = 0
for hunt3_path, hunt4_path in tqdm(test_pairs, desc="Segmenting HUNT4 GT"):
    patient_id  = os.path.basename(os.path.dirname(hunt3_path))
    gt_seg_path = os.path.join(GT_SEG_DIR, patient_id, "gt_seg.nii.gz")

    if os.path.isfile(gt_seg_path):
        n_skipped += 1
        continue

    run_fastsurfer_seg(hunt4_path, gt_seg_path)

print(f"Ground truth segmentation done. ({n_skipped} skipped)")

Segmenting HUNT4 GT: 100%|██████████| 106/106 [00:00<00:00, 108650.10it/s]

Ground truth segmentation done. (106 skipped)


## 6. Tissue Label Mapping

Map FastSurfer's DKT parcellation labels to three tissue classes using the FreeSurfer color lookup table.

In [146]:
# FreeSurfer / FastSurfer DKT parcellation → tissue class mapping
GM_LABELS = set([
    *range(1000, 1036),  # Left cortical regions (ctx-lh-*)
    *range(2000, 2036),  # Right cortical regions (ctx-rh-*)
    # Left subcortical grey matter
    10,  # Left-Thalamus-Proper
    11,  # Left-Caudate
    12,  # Left-Putamen
    13,  # Left-Pallidum
    17,  # Left-Hippocampus
    18,  # Left-Amygdala
    26,  # Left-Accumbens-area
    28,  # Left-VentralDC
    # Right subcortical grey matter
    49,  # Right-Thalamus-Proper
    50,  # Right-Caudate
    51,  # Right-Putamen
    52,  # Right-Pallidum
    53,  # Right-Hippocampus
    54,  # Right-Amygdala
    58,  # Right-Accumbens-area
    60,  # Right-VentralDC
])

WM_LABELS = set([
    2,   # Left-Cerebral-White-Matter
    41,  # Right-Cerebral-White-Matter
    7,   # Left-Cerebellum-White-Matter
    46,  # Right-Cerebellum-White-Matter
    77,  # WM-hypointensities
    85,  # Optic-Chiasm
    251, 252, 253, 254, 255,  # CC_Posterior ... CC_Anterior
])

CSF_LABELS = set([
    4,   # Left-Lateral-Ventricle
    5,   # Left-Inf-Lat-Vent
    14,  # 3rd-Ventricle
    15,  # 4th-Ventricle
    24,  # CSF
    31,  # Left-choroid-plexus
    43,  # Right-Lateral-Ventricle
    44,  # Right-Inf-Lat-Vent
    63,  # Right-choroid-plexus
    72,  # 5th-Ventricle
])


def to_binary_mask(seg_vol, label_set):
    """Convert integer segmentation volume to a binary mask for one tissue class."""
    mask = np.zeros(seg_vol.shape, dtype=bool)
    for label in label_set:
        mask |= (seg_vol == label)
    return mask

## 7. Dice Score Computation

In [147]:
def dice_coefficient(mask_pred, mask_gt):
    """Dice coefficient between two binary masks. Returns NaN if both masks are empty."""
    intersection = np.logical_and(mask_pred, mask_gt).sum()
    total = mask_pred.sum() + mask_gt.sum()
    if total == 0:
        return float("nan")
    return 2.0 * float(intersection) / float(total)

In [148]:
results = []
missing = []

for hunt3_path, _ in tqdm(test_pairs, desc="Computing Dice"):
    patient_id    = os.path.basename(os.path.dirname(hunt3_path))
    pred_seg_path = os.path.join(PRED_SEG_DIR, patient_id, "pred_seg.nii.gz")
    gt_seg_path   = os.path.join(GT_SEG_DIR,   patient_id, "gt_seg.nii.gz")

    if not os.path.isfile(pred_seg_path):
        print(f"[WARN] Missing predicted seg: {patient_id}")
        missing.append(patient_id)
        continue
    if not os.path.isfile(gt_seg_path):
        print(f"[WARN] Missing GT seg: {patient_id}")
        missing.append(patient_id)
        continue

    # np.round before int cast: NIfTI float storage can produce values like 10.0000001
    pred_seg = np.round(nib.load(pred_seg_path).get_fdata()).astype(np.int32)
    gt_seg   = np.round(nib.load(gt_seg_path).get_fdata()).astype(np.int32)

    results.append({
        "patient_id": patient_id,
        "dice_GM":    dice_coefficient(
                          to_binary_mask(pred_seg, GM_LABELS),
                          to_binary_mask(gt_seg,   GM_LABELS)),
        "dice_WM":    dice_coefficient(
                          to_binary_mask(pred_seg, WM_LABELS),
                          to_binary_mask(gt_seg,   WM_LABELS)),
        "dice_CSF":   dice_coefficient(
                          to_binary_mask(pred_seg, CSF_LABELS),
                          to_binary_mask(gt_seg,   CSF_LABELS)),
    })

df = pd.DataFrame(results)
print(f"Dice computed for {len(df)} / {len(test_pairs)} patients.")
if missing:
    print(f"Missing segmentations for {len(missing)} patients: {missing}")

Computing Dice: 100%|██████████| 106/106 [24:20<00:00, 13.78s/it]

Dice computed for 106 / 106 patients.


## 8. Results

In [150]:
print(f"=== Dice Scores — {MODEL_NAME} ===")
print(f"N patients : {len(df)}\n")

summary = df[["dice_GM", "dice_WM", "dice_CSF"]].agg(["mean", "std", "min", "max"])
summary.index = ["Mean", "Std", "Min", "Max"]
summary.columns = ["GM", "WM", "CSF"]
print(summary.to_string(float_format="{:.4f}".format))

=== Dice Scores — standard_testloss_0.0528 ===
N patients : 106

         GM     WM    CSF
Mean 0.8440 0.9208 0.9165
Std  0.0178 0.0100 0.0307
Min  0.7838 0.8892 0.7531
Max  0.8784 0.9382 0.9586


In [151]:
# Per-patient breakdown sorted by GM Dice (worst first)
df_sorted = df.sort_values("dice_GM").reset_index(drop=True)
display(df_sorted)

# Save to CSV
out_csv = f"out/{MODEL_NAME}/dice_results.csv"
df_sorted.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

,patient_id,dice_GM,dice_WM,dice_CSF
0,02507,0.783776,0.889205,0.753098
1,09902,0.795696,0.889579,0.912879
2,14098,0.797624,0.894994,0.883213
3,02217,0.799581,0.896535,0.883755
4,07823,0.804802,0.898867,0.867348
...,...,...,...,...
101,09124,0.863914,0.929210,0.952081
102,08912,0.864419,0.931783,0.867040
103,07441,0.866589,0.924638,0.923609
104,14180,0.874738,0.935152,0.947101



Saved: out/standard_testloss_0.0528/dice_results.csv
